In [3]:
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!sudo apt install ./google-chrome-stable_current_amd64.deb -y

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  119M  100  119M    0     0   179M      0 --:--:-- --:--:-- --:--:--  179M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 11.2 MB/136 MB of arc

In [4]:
!pip3 install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 11.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


- https://habr.com/ru/companies/otus/articles/596071/

In [5]:
import selenium
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

- https://habr.com/ru/articles/513966/
- https://docs.python.org/3/howto/logging.html#configuring-logging

In [6]:
import logging
from logging.config import fileConfig

import os

In [8]:
os.makedirs('logs', exist_ok=True)

fileConfig('./logs/logging_config.ini')

In [11]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import datetime

@dataclass(frozen=True)
class SteamGame:
    app_id: int = 0
    title: str = ""
    release_date: str = ""
    developer: str = ""
    additional_info: dict[str] = field(default_factory=dict)

    price: Optional[float] = None
    currency: str = "USD"

    genres: List[str] = field(default_factory=list)
    tags: List[str] = field(default_factory=list)

    @property
    def url(self) -> str:
        return f"https://steampowered.com/app/{self.app_id}"

In [17]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re

class SteamParser:
    def __init__(self, log_file='steam_parser.log', log_level=logging.INFO):
        self.driver = webdriver.Chrome(options)

        self.options = Options()
        self.options.page_load_strategy = 'eager'
        self.options.add_argument('--no-sandbox')
        self.options.add_argument('--disable-dev-shm-usage')
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")


        self.logger = logging.getLogger('steam_parser')
        # self.logger.info("Начинаем парсить")


    def parse_games_links(self, page: int):
        self.driver.get(f"https://store.steampowered.com/search/?ignore_preferences=1&page={page}")

        html = self.driver.page_source

        soup = BeautifulSoup(html, 'html.parser')

        allLinks = []

        items = soup.find_all('div', id='search_resultsRows')
        for item in items:
            links = item.find_all('a')
            for link in links:
                url = link.get('href')
                allLinks.append(url.split('?')[0].rstrip('/'))

        return allLinks

    def parse_game(self, link: str) -> SteamGame:
        self.driver.get(link)

        wait = WebDriverWait(self.driver, 20)
        reviews_container = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-featuretarget="appreviews"]'))
            )

        html = self.driver.page_source

        soup = BeautifulSoup(html, 'html.parser')

        try:
            price, currency = self.parse_game_price(soup, link)

            parsed_game = SteamGame(
                # app_id=int(link.split('/')[-1]),
                title=self.parse_name_game(soup, link),
                release_date=self.parse_release_date_game(soup),
                tags=self.parse_tags(soup),
                price=price,
                currency=currency,
            )
            return parsed_game
        except Exception as e:
            self.logger.info(f"something wrong with tags in {link}")
            print("exception game processing", e)

        return None

    def parse_tags(self, soup):
        result = []
        tags = soup.find('div', class_='glance_tags popular_tags')
        for tag in tags.find_all('a'):
            result.append(tag.text.strip())
        return result

    def float_chars(self, s: str):
      return str.isdigit(s) or s == ","


    def parse_currency(self, text: str) -> str:
      currency_map = {
          '₽': 'RUB',
          'руб.': 'RUB',
          'rub': 'RUB',
          'р.': 'RUB',
          'nt$': 'TWD',
          '$': 'USD',
          'usd': 'USD',
          '€': 'EUR',
          'eur': 'EUR',
          '£': 'GBP',
          'gbp': 'GBP',
          '¥': 'JPY',
          'jpy': 'JPY',
          '₩': 'KRW',
          'krw': 'KRW',
          'rmb': 'CNY',
          '¥': 'CNY'
      }
      currency = "USD"
      for symbol, curr in currency_map.items():
          if symbol in text:
              currency = curr
              break
      return currency


    def parse_game_price(self, soup, link: str) -> list[float, str]:
        try:
            price_elem = soup.find('div', class_='game_purchase_price price').text.strip()
            if "free" in price_elem.lower():
              return 0, "USD"

            currency = self.parse_currency(price_elem.lower())

            cleaned_price = re.sub(r'[^\d,.]', '', price_elem)

            if ',' in cleaned_price and '.' in cleaned_price:
              cleaned_price = cleaned_price.replace(',', '')
            elif ',' in cleaned_price:
              if len(cleaned_price.split(',')[-1]) == 2:
                cleaned_price = cleaned_price.replace(',', '.')
              else:
                cleaned_price = cleaned_price.replace(',', '')

            if cleaned_price == "":
              return 0, "USD"

            return float(cleaned_price), currency
        except Exception as e:
            self.logger.info(f"game {link}, has no price exception {e}")

            print(f"game {link}, has no price exception {e}")
            return 0, "USD"

    def parse_additional_description(self, soup) -> list[str]:
        price_elem = soup.find('div', class_='game_area_content_descriptors')


    def parse_name_game(self, soup: BeautifulSoup, link) -> str:
        try:
            name = soup.find('div', id='appHubAppName')
            return name.text.strip()
        except AttributeError:
            self.logger.info(f"game {link} has wrong name tag")
            return ""

    def parse_release_date_game(self, soup: BeautifulSoup) -> str:
        release_date_elem = soup.find('div', class_='release_date')
        return release_date_elem.find('div', class_='date').text.strip()

    def parse_games_on_links(self, links: list[str]) -> list[SteamGame]:
        res = []
        for i in links:
            res.append(self.parse_game(i))
        return res

In [18]:
SteamParser().parse_game("https://store.steampowered.com/app/730")

SteamGame(app_id=0, title='Counter-Strike 2', release_date='Aug 21, 2012', developer='', additional_info={}, price=0, currency='USD', genres=[], tags=['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'eSports', 'Tactical', 'First-Person', 'PvP', 'Online Co-Op', 'Co-op', 'Strategy', 'Military', 'War', 'Difficult', 'Trading', 'Realistic', 'Fast-Paced', 'Moddable'])

In [19]:
parser = SteamParser()

links = parser.parse_games_links(1)
links

['https://store.steampowered.com/app/730/CounterStrike_2',
 'https://store.steampowered.com/app/3321460/Crimson_Desert',
 'https://store.steampowered.com/app/2868840/Slay_the_Spire_2',
 'https://store.steampowered.com/app/230410/Warframe',
 'https://store.steampowered.com/app/1172470/Apex_Legends',
 'https://store.steampowered.com/app/2050650/Resident_Evil_4',
 'https://store.steampowered.com/app/2767030/Marvel_Rivals',
 'https://store.steampowered.com/app/359550/Tom_Clancys_Rainbow_Six_Siege',
 'https://store.steampowered.com/app/3764200/Resident_Evil_Requiem',
 'https://store.steampowered.com/app/2479810/Gray_Zone_Warfare',
 'https://store.steampowered.com/app/1808500/ARC_Raiders',
 'https://store.steampowered.com/app/236390/War_Thunder',
 'https://store.steampowered.com/app/252490/Rust',
 'https://store.steampowered.com/app/381210/Dead_by_Daylight',
 'https://store.steampowered.com/app/3240220/Grand_Theft_Auto_V_Enhanced',
 'https://store.steampowered.com/app/3065800/Marathon',
 'ht

In [ ]:
SteamParser().parse_games_on_links(links)